# RFW source inventory and official verification protocol

목적: 로컬 RFW 원본의 물리적 표현을 중복 없이 식별하고, 4개 그룹의 공식 10-fold 1:1 pair·landmark·이미지 목록을 검증한다.

RFW는 평가 전용 데이터다. 이 노트북은 PCA/PQ를 fit하거나 RFW를 1:N open-set 데이터로 변환하지 않는다. `real + 1.0`만 공식 프로토콜 범위이며, `dev`는 그룹×fold×genuine/impostor 층화 점검용이다.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("C:/ronbun project root could not be located")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.datasets import (
    build_rfw_verification_bundle,
    inspect_rfw_sources,
    select_rfw_protocol_scope,
    write_rfw_verification_bundle,
)

MODE = "dev"                 # "dev" 또는 "real"; 공식 결과는 real만 허용
DATA_FRACTION = 0.10          # dev pair 비율; real에서는 반드시 1.0
SEED = 42                     # dev 층화 pair 선택 seed
EXECUTE_STAGE = False         # tar 전체 검증·공식 protocol 파싱 실행 여부
WRITE_OUTPUTS = False         # 검증된 manifest를 data/interim에 기록할지 여부
VERIFY_SOURCE_SHA256 = False  # 빠른 inventory에서 대용량 파일 hash 재계산 여부

RFW_ROOT = PROJECT_ROOT / "data" / "raw" / "RFW"
JPG_ARCHIVE = RFW_ROOT / "images" / "test.tar.gz"
OUTPUT_DIR = PROJECT_ROOT / "data" / "interim" / "rfw"

if WRITE_OUTPUTS and not EXECUTE_STAGE:
    raise ValueError("WRITE_OUTPUTS=True requires EXECUTE_STAGE=True")
{
    "project_root": str(PROJECT_ROOT),
    "mode": MODE,
    "data_fraction": DATA_FRACTION,
    "execute_stage": EXECUTE_STAGE,
    "write_outputs": WRITE_OUTPUTS,
}

## 1. 물리적 소스 식별

`images/test.tar.gz`와 `bin_for_mxnet/RFW_test.tar.gz`는 같은 논리적 RFW test의 대체 표현이다. 두 파일의 이미지 수를 더하지 않는다. 아래 셀은 기본적으로 존재 여부와 크기만 읽으며 raw 데이터를 수정하지 않는다.

In [ ]:
source_inventory = inspect_rfw_sources(
    RFW_ROOT,
    project_root=PROJECT_ROOT,
    verify_sha256=VERIFY_SOURCE_SHA256,
)
{
    "summary": source_inventory.summary,
    "artifacts": [artifact.__dict__ for artifact in source_inventory.artifacts],
}

## 2. 공식 protocol 검증과 dev 범위 선택

실행 시 tar를 EOF까지 검사하고 image list, landmarks, people, pairs를 교차 검증한다. 공식 원본 bundle을 먼저 만든 뒤에만 `MODE`와 `DATA_FRACTION`을 적용한다. `dev` subset은 공식 성능으로 승격할 수 없도록 metadata가 강등된다.

In [ ]:
rfw_bundle = None
scoped_bundle = None
if EXECUTE_STAGE:
    rfw_bundle = build_rfw_verification_bundle(
        JPG_ARCHIVE,
        PROJECT_ROOT,
        strict_official=True,
    )
    scoped_bundle = select_rfw_protocol_scope(
        rfw_bundle,
        mode=MODE,
        data_fraction=DATA_FRACTION,
        seed=SEED,
    )
    protocol_summary = scoped_bundle.summary
else:
    protocol_summary = {
        "status": "not_executed",
        "next_action": "Set EXECUTE_STAGE=True and restart kernel/run all",
    }
protocol_summary

## 3. 검증 artifact 기록

BalancedFace overlap 제거에는 전체 RFW `source_identities.txt`가 필요하다. 따라서 `dev` subset이 아니라 검증된 원본 `rfw_bundle`을 기록한다. 기존 출력은 자동 덮어쓰지 않으며 `_SUCCESS`가 마지막에 생성된다.

In [ ]:
written_paths = None
if WRITE_OUTPUTS:
    if rfw_bundle is None or not rfw_bundle.summary["official_protocol_validated"]:
        raise RuntimeError("A strict official RFW bundle is required before writing")
    written_paths = write_rfw_verification_bundle(
        rfw_bundle,
        OUTPUT_DIR,
        overwrite=False,
    )
    written_paths = {name: str(path) for name, path in written_paths.items()}
written_paths or "WRITE_OUTPUTS=False: no files written"

## 다음 단계

1. 이 노트북을 `EXECUTE_STAGE=True`, `WRITE_OUTPUTS=True`로 커널 재시작 후 전체 실행한다.
2. `data/interim/rfw/_SUCCESS`와 `source_identities.txt`를 확인한다.
3. `notebooks/balancedface/data_preparation.ipynb`에서 중복 identity 제거와 개발·보정 분할을 수행한다.

현재 MS1MV2로 표기된 checkpoint는 RFW headline 평가 자격이 없다. protocol manifest 생성과 모델 평가 자격은 별개의 단계다.